In [52]:
import pandas as pd
import warnings 
warnings.filterwarnings("ignore")

# Control Variable(고객데이터) 추가하기
1. 데이터 호출
2. 기본 고객정보 코딩
2. 데이터 결합


### 1. 데이터 호출
* 분석용데이터, 고객정보, 1년간고객구매이력
* 데이터 결합을 위한 INCS_NO 필요   

### 2. 고객 기본정보 코딩
* 남:0, 여:1
* age - 10,20대, 30대, 40대, 50대, 60대이상
* groupby를 이용해 고객별 연평균구매액 코딩
* 고객정보 + 구매이력 결합

### 3. 데이터 결합
* 분석용데이터 + 고객정보 결합
* join 후 누락된 INCS_NO가 많은지 확인 필요
* 누락된 데이터가 많다면 데이터 새로 다운받아야 할 듯 함

### 1. 데이터 호출

In [62]:
df = pd.read_csv('./data/sess_cutoff.csv', index_col=0)
print(df.shape)
df.head(2)

(33479, 11)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,INCS_NO,LST_SESS_TIME,APP,AVG_DEPTH,day_off
39,001ef1290ae23824ae5e97a9fa2a00f31719788953,42,12,1,42,139080,4acc3c2221d10da113ac56989dba4b615457c61195a1a8...,2024-07-01 08:24:21.803,1,13.571429,0
40,001ef1290ae23824ae5e97a9fa2a00f31719926158,5,1,0,5,0,4acc3c2221d10da113ac56989dba4b615457c61195a1a8...,2024-07-02 22:16:40.917,1,13.333333,1


In [63]:
cust = pd.read_csv('./raw/CUST_INFO.csv')
print(cust.shape)
cust.head(2)

(966941, 3)


,INCS_NO,AGE,SEX_CD
0,e83ec27d1ea0bdfbee8296ca86cab339fa38be7e704145...,33,F
1,5888134982e132a44ca560752ebdb604521f71573c8466...,67,F


In [64]:
cust_sal = pd.read_csv('./raw/CUST_SAL.csv')
cust_sal['STND_YMD'] = pd.to_datetime(cust_sal['STND_YMD'])
print(cust_sal.shape)
cust_sal.head(2)

(2868414, 3)


,INCS_NO,POS_NET_PRD_SAL_AMT,STND_YMD
0,857c1e5d32cac4c06011e4c9297e7cd05f6f8629288bd1...,32850.0,2024-02-21
1,2df11ad7837d63b48fed4bd6db79a932d74239c703f9b1...,23800.0,2024-02-21


### 2. 고객 기본정보 코딩

In [65]:
cust_sal['YEAR'] = cust_sal['STND_YMD'].dt.year 
cust_sal_avg = cust_sal.groupby(['INCS_NO', 'YEAR']).POS_NET_PRD_SAL_AMT.mean().reset_index() 
cust_sal_avg = cust_sal_avg.rename(columns={'POS_NET_PRD_SAL_AMT': 'AVG_SAL_AMT'})
cust_sal_avg = cust_sal_avg.groupby(['INCS_NO']).AVG_SAL_AMT.mean().reset_index()

cust_sal_avg.head(2)

,INCS_NO,AVG_SAL_AMT
0,0000a9677f34fb28ae920f7346600a59c9ef3e2a65c347...,0.000000
1,0000e30a4703362d8d858cf0b777871baa3c392562f3d1...,5566.666667


In [66]:
cust = pd.merge(cust, cust_sal_avg, how='left', on='INCS_NO')
print(cust.shape)
cust.head(2)

(966941, 4)


,INCS_NO,AGE,SEX_CD,AVG_SAL_AMT
0,e83ec27d1ea0bdfbee8296ca86cab339fa38be7e704145...,33,F,NaN
1,5888134982e132a44ca560752ebdb604521f71573c8466...,67,F,NaN


In [67]:
cust[cust['AVG_SAL_AMT'].notna()].shape

(253882, 4)

### 3. 데이터 결합

In [68]:
temp = pd.merge(df, cust, how='left', on='INCS_NO')
print(temp.shape)
temp.head(2)

(33479, 14)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,INCS_NO,LST_SESS_TIME,APP,AVG_DEPTH,day_off,AGE,SEX_CD,AVG_SAL_AMT
0,001ef1290ae23824ae5e97a9fa2a00f31719788953,42,12,1,42,139080,4acc3c2221d10da113ac56989dba4b615457c61195a1a8...,2024-07-01 08:24:21.803,1,13.571429,0,48.0,F,NaN
1,001ef1290ae23824ae5e97a9fa2a00f31719926158,5,1,0,5,0,4acc3c2221d10da113ac56989dba4b615457c61195a1a8...,2024-07-02 22:16:40.917,1,13.333333,1,48.0,F,NaN


In [69]:
# cust가 누락된 데이터 수 확인
# 누락된 데이터가 너무 많다면 cust 데이터 재검토 필요

print(temp[temp['AGE'].isna()].shape)
print(temp[temp['AVG_SAL_AMT'].isna()].shape)

(7362, 14)
(17203, 14)


In [70]:
temp = temp[temp['AGE'].notna()]
cv_df = temp[temp['AVG_SAL_AMT'].notna()]
print(cv_df.shape)
cv_df.head(2)

(16276, 14)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,INCS_NO,LST_SESS_TIME,APP,AVG_DEPTH,day_off,AGE,SEX_CD,AVG_SAL_AMT
7,00321C571EF64FCC85ECA0FF1A5F562B1721178358,9,4,1,9,42000,041cdf8ebd35ea26ae0e39da46b6e697d45d51e3a69204...,2024-07-17 10:07:16.029,1,13.333333,0,43.0,F,32300.000000
9,003adf3e2d2ff7f800069fa65b71a6731722240453,9,4,1,9,12000,b2cb86c304e8959316302586276c159f095fe5de6b71c8...,2024-07-29 17:08:23.674,1,25.000000,0,59.0,F,9686.111111


In [48]:
cv_df['INCS_NO'].nunique()

31432

In [49]:
def age_coding(age):
    if age < 30:
        return 1020
    elif age < 40:
        return 30
    elif age < 50:
        return 40
    elif age < 60:
        return 50
    else:
        return 60

def sex_coding(sex):
    if sex == 'F':
        return 1
    else:
        return 0

In [50]:
# 나이 원핫인코딩
dummy_df = cv_df.copy()
dummy_df['AGE'] = dummy_df['AGE'].apply(age_coding)
dummy_df['SEX_CD'] = dummy_df['SEX_CD'].apply(sex_coding)
dummy_df = pd.get_dummies(dummy_df, columns=['AGE'], dtype=int)

# 40대 데이터 제거
dummy_df = dummy_df.drop(columns=['AGE_40'])
print(dummy_df.shape)
dummy_df.head(2)

(107340, 15)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,INCS_NO,LST_SESS_TIME,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020
0,000211c41b2c524ea758d997901733fe1720001995,5,0,-1,5,0,1d061118266aaac746e4418b030d8d10046fd4a4adabe0...,2024-07-03 19:21:01.213,1,1,9100.0,0,1,0,0
1,000211c41b2c524ea758d997901733fe1720594007,8,2,1,8,22320,1d061118266aaac746e4418b030d8d10046fd4a4adabe0...,2024-07-10 15:49:36.741,0,1,9100.0,0,1,0,0


In [51]:
dummy_df.to_csv('./data/cv_control.csv')

In [29]:
dummy_df['INCS_NO'].nunique()

4637